# Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from uavloc.data.annotated import IGNORE_LABEL
from uavloc.features.segmentation import CLASSES, load_segmenter
from uavloc.plotting.correlation import SEG_PALETTE
from uavloc.track import REPO_ROOT

# Params

In [ ]:
HERE = REPO_ROOT / "pocs" / "lraspp-pi5"
MODEL_DIR = HERE / "models"
RUN_ID = "pi5-masks"
MODELS = ["drone", "satellite"]
RUNTIMES = ["onnxruntime", "torch"]
IGNORE_RGB = [255, 255, 255]
DISAGREE_RGB = [255, 0, 255]
PANEL_IN = 3.0

# Data

In [ ]:
samples = {name: dict(np.load(MODEL_DIR / f"sample-{name}.npz")) for name in MODELS}
pi_logits = dict(np.load(HERE / "runs" / "logs" / RUN_ID / "pi-logits.npz"))
sorted(pi_logits)

# Model

In [ ]:
@torch.no_grad()
def mac_logits(name: str) -> np.ndarray:
    model = load_segmenter(MODEL_DIR / f"lraspp-{name}.pt", "cpu", architecture="lraspp")
    return model(torch.from_numpy(samples[name]["x"])).numpy()

In [ ]:
def mask_rgb(mask: np.ndarray) -> np.ndarray:
    """Class mask `(H, W)` → palette RGB; IGNORE_LABEL pixels white."""
    return np.where((mask == IGNORE_LABEL)[..., None], IGNORE_RGB, SEG_PALETTE[np.minimum(mask, len(CLASSES) - 1)])

In [ ]:
def compare_row(name: str, mac: np.ndarray) -> dict:
    mac_mask = mac[0].argmax(0)
    row = {"model": name}
    for runtime in RUNTIMES:
        pi = pi_logits[f"{runtime}-{name}"]
        row[f"{runtime}_agreement"] = float((pi[0].argmax(0) == mac_mask).mean())
        row[f"{runtime}_max_abs_diff"] = float(np.abs(pi - mac).max())
    return row

# Run

In [ ]:
mac = {name: mac_logits(name) for name in MODELS}

In [ ]:
pd.DataFrame([compare_row(name, mac[name]) for name in MODELS]).set_index("model")

# Plots

In [ ]:
def plot_masks(name: str) -> plt.Figure:
    mac_mask = mac[name][0].argmax(0)
    pi_masks = {runtime: pi_logits[f"{runtime}-{name}"][0].argmax(0) for runtime in RUNTIMES}
    disagree = np.any([m != mac_mask for m in pi_masks.values()], axis=0)
    panels = {"image": samples[name]["rgb"], "label": mask_rgb(samples[name]["label"]), "Mac torch": mask_rgb(mac_mask),
              **{f"Pi {runtime}": mask_rgb(m) for runtime, m in pi_masks.items()},
              f"Pi ≠ Mac ({disagree.sum()} px)": np.where(disagree[..., None], DISAGREE_RGB, samples[name]["rgb"] // 3)}
    fig, axes = plt.subplots(1, len(panels), figsize=(PANEL_IN * len(panels), PANEL_IN))
    for ax, (title, img) in zip(axes, panels.items()):
        ax.imshow(img.astype(np.uint8))
        ax.set_title(title)
        ax.axis("off")
    fig.suptitle(name)
    fig.tight_layout()
    return fig

In [ ]:
plot_masks("drone");

In [ ]:
plot_masks("satellite");